# Tideway Topology API Examples

Helper methods paired with direct calls for topology endpoints: graphs, node lookups by criteria, node kinds, and visualization state. Reference: https://docs.bmc.com/xwiki/bin/view/IT-Operations-Management/Discovery/BMC-Helix-Discovery/DAAS/Integrating/Using-the-REST-APIs/Endpoints-in-the-REST-API/.

## Setup
- Install tideway (e.g. `pip install -e .` from repo root).
- Set `TARGET` and `TOKEN` below. Do **not** commit secrets.
- Adjust ids, filters, and bodies per your environment.

In [ ]:

import tideway

# Configuration
TARGET = 'appliance-hostname-or-ip'  # e.g. 'discovery.example.com'
TOKEN = 'your-api-token'            # keep secrets out of source control
API_VERSION = '1.16'                # latest supported API
SSL_VERIFY = False                  # set to True when using valid certs

tw = tideway.appliance(TARGET, TOKEN, api_version=API_VERSION, ssl_verify=SSL_VERIFY)
topology = tw.topology()

def show_response(resp, label):
    if resp.ok:
        try:
            return resp.json()
        except Exception:
            return resp.text
    try:
        body = resp.json()
    except Exception:
        body = resp.text
    return {'endpoint': label, 'status': resp.status_code, 'body': body}

tw.api_about.json()  # quick sanity check


## Node graph by node id
Helper vs direct GET for `/data/nodes/{id}/graph` with focus/apply_rules/complete flags.

In [ ]:

node_id = ''  # e.g. '1234567890abcdef'
focus = 'software-connected'
apply_rules = True
complete = False

if node_id:
    graph_helper = topology.get_data_nodes_graph(node_id, focus=focus, apply_rules=apply_rules, complete=complete)
    graph_direct = topology.get(
        f"/data/nodes/{node_id}/graph?focus={focus}&apply_rules={str(apply_rules).lower()}&complete={str(complete).lower()}"
    )
    graph_calls = {
        f"/data/nodes/{node_id}/graph (helper)": show_response(graph_helper, f"/data/nodes/{node_id}/graph"),
        f"/data/nodes/{node_id}/graph (direct)": show_response(graph_direct, f"/data/nodes/{node_id}/graph"),
    }
else:
    graph_calls = 'Set node_id to fetch a topology graph.'
graph_calls


## Topology nodes (criteria search)
Post criteria to `/topology/nodes` using helper vs direct.

In [ ]:

topology_nodes_body = {
    # 'criteria': {'kind': 'Host', 'ip_address': '10.0.0.1'},
    # 'attributes': ['name', 'kind'],
}

if topology_nodes_body:
    topo_nodes_helper = topology.post_topology_nodes(topology_nodes_body)
    topo_nodes_direct = topology.post('/topology/nodes', topology_nodes_body)
    topo_nodes_calls = {
        '/topology/nodes (helper POST)': show_response(topo_nodes_helper, '/topology/nodes'),
        '/topology/nodes (direct POST)': show_response(topo_nodes_direct, '/topology/nodes'),
    }
else:
    topo_nodes_calls = 'Set topology_nodes_body to post /topology/nodes.'
topo_nodes_calls


## Topology nodes by kinds
Post kind filters to `/topology/nodes/kinds` using helper vs direct.

In [ ]:

topology_kinds_body = {
    # 'kinds': ['Host', 'SoftwareInstance'],
    # 'attributes': ['name', 'kind'],
}

if topology_kinds_body:
    topo_kinds_helper = topology.post_topology_nodes_kinds(topology_kinds_body)
    topo_kinds_direct = topology.post('/topology/nodes/kinds', topology_kinds_body)
    topo_kinds_calls = {
        '/topology/nodes/kinds (helper POST)': show_response(topo_kinds_helper, '/topology/nodes/kinds'),
        '/topology/nodes/kinds (direct POST)': show_response(topo_kinds_direct, '/topology/nodes/kinds'),
    }
else:
    topo_kinds_calls = 'Set topology_kinds_body to post /topology/nodes/kinds.'
topo_kinds_calls


## Visualization state
Get or update visualization state via helper vs direct calls.

In [ ]:

# GET visualization state
viz_helper = topology.get_topology_viz_state()
viz_direct = topology.get('/topology/visualization_state')

viz_calls = {
    '/topology/visualization_state (helper)': show_response(viz_helper, '/topology/visualization_state'),
    '/topology/visualization_state (direct)': show_response(viz_direct, '/topology/visualization_state'),
}

viz_patch_body = {
    # 'layout': {},
    # 'filters': {},
}

viz_put_body = {
    # 'layout': {},
    # 'filters': {},
}

if viz_patch_body:
    viz_patch_helper = topology.patch_topology_viz_state(viz_patch_body)
    viz_patch_direct = topology.patch('/topology/visualization_state', viz_patch_body)
    viz_calls['/topology/visualization_state (helper PATCH)'] = show_response(viz_patch_helper, '/topology/visualization_state')
    viz_calls['/topology/visualization_state (direct PATCH)'] = show_response(viz_patch_direct, '/topology/visualization_state')

if viz_put_body:
    viz_put_helper = topology.put_topology_viz_state(viz_put_body)
    viz_put_direct = topology.put('/topology/visualization_state', viz_put_body)
    viz_calls['/topology/visualization_state (helper PUT)'] = show_response(viz_put_helper, '/topology/visualization_state')
    viz_calls['/topology/visualization_state (direct PUT)'] = show_response(viz_put_direct, '/topology/visualization_state')

viz_calls
